In [ ]:
import clickhouse_connect
import pandas as pd
import os

pd.options.display.max_colwidth = 100

# Вбейте свой телеграм никнейм, чтобы в случае проблем мы могли вас индефицировать
database = 'SkubaM'

client = clickhouse_connect.get_client(host='clickhouse01', port=8123, username=os.getenv('CLICKHOUSE_USER'), password=os.getenv('CLICKHOUSE_PASSWORD'))

In [ ]:
client.command('''
    drop DATABASE IF EXISTS SkubaM ON CLUSTER '{cluster}';
''')

In [ ]:
client.command('''
    CREATE DATABASE IF NOT EXISTS SkubaM ON CLUSTER '{cluster}';
''')

In [ ]:
client.command('''
    CREATE TABLE IF NOT EXISTS SkubaM.events_local
ON CLUSTER '{cluster}'
(
    id UInt64,
    user_id UInt64,
    event_type LowCardinality(String),
    payload String,
    updated_at DateTime
)
ENGINE = ReplicatedReplacingMergeTree(
    '/clickhouse/tables/{shard}/SkubaM/events_local',
    '{replica}',
    updated_at
)
PARTITION BY toYYYYMM(updated_at)
ORDER BY (id)
''')

In [ ]:
client.command('''
    CREATE TABLE IF NOT EXISTS SkubaM.events
    ON CLUSTER '{cluster}'
    AS SkubaM.events_local
    ENGINE = Distributed('{cluster}', 'SkubaM', 'events_local', cityHash64(id));
''')

In [ ]:
client.command('''
    INSERT INTO SkubaM.events (id, user_id, event_type, payload, updated_at) VALUES
    (1, 10, 'view',  '{"page":"home"}',  now()),
    (2, 10, 'click', '{"btn":"buy"}',    now()),
    (1, 10, 'view',  '{"page":"home2"}', now() + 10);
 ''')

In [ ]:
client.query_df('''
    select * from SkubaM.events_local
''')

In [ ]:
client.query_df('''
    select * from SkubaM.events
''')

In [ ]:
client.command('''
    INSERT INTO SkubaM.events (id, user_id, event_type, payload, updated_at) VALUES
    (1, 10, 'view',  '{"page":"home"}',  now()),
    (2, 10, 'click', '{"btn":"buy"}',    '2026-02-26 15:19:12'),
    (1, 10, 'view',  '{"page":"home2"}', now() + 10);
 ''')

In [ ]:
client.query_df('''
    select * from SkubaM.events
''')

In [ ]:
client.query_df('''
    select * from SkubaM.events_local
''')